In [ ]:
!pip install spacy

In [2]:
!python -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 2.8 MB/s eta 0:00:05
     ----- ---------------------------------- 1.8/12.8 MB 5.6 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 5.6 MB/s eta 0:00:02
     ----------------- ---------------------- 5.5/12.8 MB 6.6 MB/s eta 0:00:02
     ---------------------- ----------------- 7.3/12.8 MB 7.1 MB/s eta 0:00:01
     ----------------------------- ---------- 9.4/12.8 MB 7.4 MB/s eta 0:00:01
     ---------------------------------- ----- 11.0/12.8 MB 7.5 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 7.5 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 7.4 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import re
import pandas as pd
# Load the uploaded CSV file
file_path = 'Data.csv'
df = pd.read_csv(file_path)

# Display the first few rows and the column names to understand the structure
df.head(), df.columns

(                                                Data
 0  Watch or listen live weekdays at 8:30am MT at ...
 1  Watch or listen live weekdays at 8:30am MT at ...
 2               Chubby And Hot, Always Stir The Pot!
 3               Chubby And Hot, Always Stir The Pot!
 4  Journalist, publisher of Rebel News — telling ...,
 Index(['Data'], dtype='object'))

In [4]:

# Show full column content when printing
pd.set_option('display.max_colwidth', None)  # Removes truncation of long strings

# Define regex pattern to match decimal numbers like "5."
pattern_5_dot = r'\b\d+\.\b'

# Find matching rows
match_5_dot = df[df['Data'].str.contains(pattern_5_dot, regex=True)]

# Display matching records
print("Matching records:\n")
print(match_5_dot)

# Display row indices
print("\nLine numbers (0-based index) with '5.':")
print(match_5_dot.index.tolist())


Matching records:

                                                                                                                                                                   Data
473                                       City of Edmonton News & Updates. Active weekdays 8am-4:30pm. Report concerns: 311.edmonton.ca. Edmonton Transit @takeETSalert
487                                       City of Edmonton News & Updates. Active weekdays 8am-4:30pm. Report concerns: 311.edmonton.ca. Edmonton Transit @takeETSalert
2388         Don Hladiuk is Calgary's eye on astronomy & space science news. On the CBC EyeOpener 1010AM or 99.1FM on the 1st or 2nd Monday of the month at 7:36 AM MT.
2718         Don Hladiuk is Calgary's eye on astronomy & space science news. On the CBC EyeOpener 1010AM or 99.1FM on the 1st or 2nd Monday of the month at 7:36 AM MT.
3208                                     AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest 

In [5]:
duplicate_count = df.duplicated().sum()
print("Number of completely duplicate rows:", duplicate_count)


Number of completely duplicate rows: 5353


In [6]:
text_duplicates = df['Data'].duplicated().sum()
print("Number of duplicate 'Data' entries:", text_duplicates)


Number of duplicate 'Data' entries: 5353


In [7]:
duplicated_rows = df[df['Data'].duplicated(keep=False)]
duplicated_rows.head()


,Data
0,Watch or listen live weekdays at 8:30am MT at ryanjespersen.com. Subscribe via YouTube or your favourite podcast app. #RealTalkRJ
1,Watch or listen live weekdays at 8:30am MT at ryanjespersen.com. Subscribe via YouTube or your favourite podcast app. #RealTalkRJ
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
6,Albertan. Mom to children & dogs. Wife. Friend. Leader of the Alberta NDP. Account run by Rachel Notley and staff. She/Her #ableg


In [8]:
df_no_duplicates = df.drop_duplicates(subset='Data').copy()


These are dates, not decimal numbers.

You may want to exclude dates from decimal detection if accuracy is important for numerical feature extraction.

Standard decimals: 12.5, .75

Numbers ending with a dot: 5.

Comma-formatted decimals: 5,000.25

Scientific notation: 1.2e3, 2E5


Pre-tag or remove timestamps/dates with regex

In [20]:


# Regex patterns (as you have)
time_pattern = r'\b\d{1,2}:\d{2}(?:[ap]m)?\b'
date_pattern = r'\b\d{1,2}[./-]\d{1,2}[./-]?\d{0,4}\b'
year_range_pattern = r'\b\d{4}[-–]\d{2,4}\b'
apostrophe_year = r"\b'\d{2}\b"

combined_pattern = f'({time_pattern}|{date_pattern}|{year_range_pattern}|{apostrophe_year})'

# Use .loc for assignment to avoid SettingWithCopyWarning
df_no_duplicates.loc[:, 'Cleaned_Data'] = df_no_duplicates['Data'].str.replace(combined_pattern, '', regex=True)

decimal_pattern = r'\b\d*\.\d+\b|\b\d+\.\b|\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\.?\d*[eE][+-]?\d+\b'
df_no_duplicates.loc[:, 'Decimals'] = df_no_duplicates['Cleaned_Data'].str.findall(decimal_pattern)

print(df_no_duplicates[['Data', 'Decimals']].head())



                                                                                                                                                            Data  \
0                              Watch or listen live weekdays at 8:30am MT at ryanjespersen.com. Subscribe via YouTube or your favourite podcast app. #RealTalkRJ   
2                                                                                                                           Chubby And Hot, Always Stir The Pot!   
4      Journalist, publisher of Rebel News — telling the other side of the story. Awarded the Queen’s Diamond Jubilee Medal for advancing freedom of expression.   
5  Edmonton City Councillor for #WardMétis. #YEG Urban Planner, Sustainability & Sociologist. Founder. Cities, climate action, health, inclusion. She/her #yegcc   
6                              Albertan. Mom to children & dogs. Wife. Friend. Leader of the Alberta NDP. Account run by Rachel Notley and staff. She/Her #ableg   

  Decimals  
0 

 Use Named Entity Recognition (NER)

In [21]:
import spacy

# Load English NER model
nlp = spacy.load("en_core_web_sm")

def remove_dates_and_times(text):
    doc = nlp(text)
    cleaned = ''.join([
        token.text_with_ws for token in doc
        if token.ent_type_ not in ["DATE", "TIME"]
    ])
    return cleaned

# Safely apply to dataset
df_no_duplicates.loc[:, 'NER_Cleaned'] = df_no_duplicates['Data'].apply(remove_dates_and_times)

# Apply decimal regex after NER cleaning (use .loc)
df_no_duplicates.loc[:, 'NER_Decimals'] = df_no_duplicates['NER_Cleaned'].str.findall(decimal_pattern)


In [22]:
# Regex pattern to match comma-formatted decimals like "5,000.25"
pattern = r'\b\d{1,3}(?:,\d{3})+\.\d+\b'

# Filter rows that match the pattern
matches = df_no_duplicates[df_no_duplicates['Data'].str.contains(pattern, regex=True)]

# Print results
print("Total matches:", len(matches))
print(matches[['Data']])

Total matches: 0
Empty DataFrame
Columns: [Data]
Index: []


In [23]:
# Regex pattern to match scientific notation (e.g., 1.2e3, 2E5, 3.0E-2)
pattern = r'\b\d+\.?\d*[eE][+-]?\d+\b'

# Filter rows that match the pattern
matches = df_no_duplicates[df_no_duplicates['Data'].str.contains(pattern, regex=True)]

# Print results
print("Total matches:", len(matches))
print(matches[['Data']])

Total matches: 0
Empty DataFrame
Columns: [Data]
Index: []


In [24]:
# Make sure these patterns are defined before combining!
time_pattern = r'\b\d{1,2}:\d{2}(?:[ap]m)?\b'
date_pattern = r'\b\d{1,2}[./-]\d{1,2}(?:[./-]\d{2,4})?\b'
year_range_pattern = r'\b\d{4}[-–]\d{2,4}\b'
apostrophe_year = r"\b'\d{2}\b"
phone_pattern = r'\+?\d{1,2}\.\d{3}\.\d{3}\.\d{4}'

# Combine all patterns into one
combined_pattern = f"({time_pattern}|{date_pattern}|{year_range_pattern}|{apostrophe_year}|{phone_pattern})"

# Clean those patterns out of the text
df_no_duplicates.loc[:, 'Cleaned_Data'] = df_no_duplicates['Data'].str.replace(combined_pattern, '', regex=True)

# Final extended decimal regex:
decimal_pattern = r'''
    \b\d*\.\d+\b                    # standard decimal like .75, 12.3
    | \b\d{1,3}(?:,\d{3})+\.\d+\b   # comma formatted decimal like 5,000.25
    | \b\d+\.?\d*[eE][+-]?\d+\b     # scientific notation like 1.2e3, 2E5
'''

# Check which rows contain a decimal
contains_decimal = df_no_duplicates['Cleaned_Data'].str.contains(decimal_pattern, regex=True, flags=re.VERBOSE)

# Count matching records
decimal_count = contains_decimal.sum()
print(f"Number of records containing a decimal number: {decimal_count}")



Number of records containing a decimal number: 3


In [25]:
# Display the 17 matching records
matching_rows = df_no_duplicates[contains_decimal]
print(matching_rows[['Data']])


                                                                                                                                                                 Data
3208                                   AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC
4568  #openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: 3.2.\n#LULC #landscape #soil #urban #climate
5113                                                                                                  Playing Southern Saskatchewan's Best Country! Country 100.7 FM!


In [26]:
# Regex to find decimal numbers
decimal_pattern = r'\b\d+\.\d+\b'

# Count records containing decimal numbers
decimal_count = df_no_duplicates['Data'].str.contains(decimal_pattern, regex=True).sum()

print("Number of records with a decimal number:", decimal_count)
# Filter rows with decimal numbers
decimal_rows = df_no_duplicates[df_no_duplicates['Data'].str.contains(r'\b\d+\.\d+\b', regex=True)]

# Display them
decimal_rows

Number of records with a decimal number: 7


,Data,Cleaned_Data,Decimals,NER_Cleaned,NER_Decimals
3208,AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC,AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC,[106.1],AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC,[106.1]
3663,hier schreiben die Hühner aus dem Stall (*25./26.4. 2019) . \nKörner und Würmer immer willkommen,hier schreiben die Hühner aus dem Stall (*25./. 2019) . \nKörner und Würmer immer willkommen,[25],hier schreiben die Hühner aus dem Stall (*) . \nKörner und Würmer immer willkommen,[]
3682,"Providing RV repair, parts & accessories, 24/7 mobile service, custom rebuilds, rentals & quality advice from Red Seal Certified RV Technicians. +1.587.989.1271","Providing RV repair, parts & accessories, mobile service, custom rebuilds, rentals & quality advice from Red Seal Certified RV Technicians.",[989.1271],"Providing RV repair, parts & accessories, 24/7 mobile service, custom rebuilds, rentals & quality advice from Red Seal Certified RV Technicians. +1.587.989.1271","[24, 7, 1.587, .989, .1271]"
4568,"#openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: 3.2.\n#LULC #landscape #soil #urban #climate","#openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: .\n#LULC #landscape #soil #urban #climate",[],"#openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF : 3.905; CiteScore : 3.2.\n#LULC #landscape #soil #urban #climate","[3.905, 3.2]"
4626,"Hoping hardcore twitter 2.0 lets me express honest opinions about Danielle Smith's supporters.\n\n""Let them see what is on the end of that long newspaper spoon.""","Hoping hardcore twitter lets me express honest opinions about Danielle Smith's supporters.\n\n""Let them see what is on the end of that long newspaper spoon.""",[],"Hoping hardcore twitter 2.0 lets me express honest opinions about Danielle Smith's supporters.\n\n""Let them see what is on the end of that long newspaper spoon.""",[2.0]
5113,Playing Southern Saskatchewan's Best Country! Country 100.7 FM!,Playing Southern Saskatchewan's Best Country! Country 100.7 FM!,[100.7],Playing Southern Saskatchewan's Best Country! Country 100.7 FM!,[100.7]
5244,96.5 CKFM - All Hit Country!\n📻 96.5 FM \n🌐 ckfm.ca \nYour source for everything happening in Mountain View County!,CKFM - All Hit Country!\n📻 FM \n🌐 ckfm.ca \nYour source for everything happening in Mountain View County!,[],96.5 CKFM - All Hit Country!\n📻 96.5 FM \n🌐 ckfm.ca \nYour source for everything happening in Mountain View County!,"[96.5, 96.5]"


In [29]:
# Step 1: Define cleanup patterns
time_pattern = r'\b\d{1,2}:\d{2}(?:[ap]m)?\b'
date_pattern = r'\b\d{1,2}[./-]\d{1,2}(?:[./-]\d{2,4})?\b'
year_range_pattern = r'\b\d{4}[-–]\d{2,4}\b'
apostrophe_year = r"\b'\d{2}\b"
phone_pattern = r'\+?\d{1,2}\.\d{3}\.\d{3}\.\d{4}'

# Combine all patterns
combined_pattern = f"({time_pattern}|{date_pattern}|{year_range_pattern}|{apostrophe_year}|{phone_pattern})"

# Step 2: Clean the Data column using .loc to avoid SettingWithCopyWarning
df_no_duplicates.loc[:, 'Cleaned_Data'] = df_no_duplicates['Data'].str.replace(combined_pattern, '', regex=True)

# Step 3: Define decimal detection pattern (excluding cleaned cases)
decimal_pattern = r'''
    \b\d*\.\d+\b                    # standard decimal like .75, 12.3
    | \b\d{1,3}(?:,\d{3})+\.\d+\b   # comma formatted decimal like 5,000.25
    | \b\d+\.?\d*[eE][+-]?\d+\b     # scientific notation like 1.2e3, 2E5
'''

# Step 4: Detect decimals in cleaned data
mask = df_no_duplicates['Cleaned_Data'].str.contains(decimal_pattern, regex=True, flags=re.VERBOSE)
decimal_count = mask.sum()
matching_rows = df_no_duplicates[mask]

# Step 5: Output the results
print("✅ Number of records with decimal numbers after cleaning:", decimal_count)
print("\n✅ Matching records:")
print(matching_rows[['Data']])




✅ Number of records with decimal numbers after cleaning: 3

✅ Matching records:
                                                                                                                                                                 Data
3208                                   AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC
4568  #openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: 3.2.\n#LULC #landscape #soil #urban #climate
5113                                                                                                  Playing Southern Saskatchewan's Best Country! Country 100.7 FM!


In [31]:
# Step 1: Define cleanup patterns (excluding date to preserve decimals like 96.5)
time_pattern = r'\b\d{1,2}:\d{2}(?:[ap]m)?\b'
year_range_pattern = r'\b\d{4}[-–]\d{2,4}\b'
apostrophe_year = r"\b'\d{2}\b"
phone_pattern = r'\+?\d{1,2}\.\d{3}\.\d{3}\.\d{4}(?!\d)'

# Step 2: Combine only safe cleaning patterns
combined_pattern = f"({time_pattern}|{year_range_pattern}|{apostrophe_year}|{phone_pattern})"

# Step 3: Clean the Data column - FIXED with .loc to prevent warnings
df_no_duplicates.loc[:, 'Cleaned_Data'] = df_no_duplicates['Data'].str.replace(combined_pattern, '', regex=True)

# Step 4: Define decimal detection pattern (standard, comma, scientific)
decimal_pattern = r'''
    \b\d*\.\d+\b                    # .75, 12.3
    | \b\d{1,3}(?:,\d{3})+\.\d+\b   # 5,000.25
    | \b\d+\.?\d*[eE][+-]?\d+\b     # 1.2e3, 2E5
'''

# Step 5: Detect decimals in cleaned data (no warning needed here, only for assignments)
mask = df_no_duplicates['Cleaned_Data'].str.contains(decimal_pattern, regex=True, flags=re.VERBOSE)
decimal_count = mask.sum()
matching_rows = df_no_duplicates[mask]

# Step 6: Output the results
print("✅ Number of records with decimal numbers after cleaning:", decimal_count)
print("\n✅ Matching records:")
print(matching_rows[['Data']])



✅ Number of records with decimal numbers after cleaning: 6

✅ Matching records:
                                                                                                                                                                   Data
3208                                     AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC
3663                                                                  hier schreiben die Hühner aus dem Stall  (*25./26.4. 2019) . \nKörner und Würmer immer willkommen
4568    #openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: 3.2.\n#LULC #landscape #soil #urban #climate
4626  Hoping hardcore twitter 2.0 lets me express honest opinions about Danielle Smith's supporters.\n\n"Let them see what is on the end of that long newspaper spoon."
5113                                                                            

The date pattern is included, when we try to reomve, it removes the real values like 96.5 hence needed trying to create a custom date pattern.

In [32]:
# Step 1: Define cleanup patterns (excluding date to preserve decimals like 96.5)
time_pattern = r'\b\d{1,2}:\d{2}(?:[ap]m)?\b'
year_range_pattern = r'\b\d{4}[-–]\d{2,4}\b'
apostrophe_year = r"\b'\d{2}\b"
phone_pattern = r'\+?\d{1,2}\.\d{3}\.\d{3}\.\d{4}(?!\d)'

# Step 2: Combine only safe cleaning patterns
combined_pattern = f"({time_pattern}|{year_range_pattern}|{apostrophe_year}|{phone_pattern})"

# Step 3: Clean the Data column - FIXED with .loc to prevent warnings
df_no_duplicates.loc[:, 'Cleaned_Data'] = df_no_duplicates['Data'].str.replace(combined_pattern, '', regex=True)

# Step 4: Define decimal detection pattern (standard, comma, scientific)
decimal_pattern = r'''
    \b\d*\.\d+\b                    # .75, 12.3
    | \b\d{1,3}(?:,\d{3})+\.\d+\b   # 5,000.25
    | \b\d+\.?\d*[eE][+-]?\d+\b     # 1.2e3, 2E5
'''

# Step 5: Detect decimals in cleaned data (no warning needed here, only for assignments)
mask = df_no_duplicates['Cleaned_Data'].str.contains(decimal_pattern, regex=True, flags=re.VERBOSE)
decimal_count = mask.sum()
matching_rows = df_no_duplicates[mask]

# Step 6: Output the results
print("✅ Number of records with decimal numbers after cleaning:", decimal_count)
print("\n✅ Matching records:")
print(matching_rows[['Data']])


✅ Number of records with decimal numbers after cleaning: 6

✅ Matching records:
                                                                                                                                                                   Data
3208                                     AIR 106.1 FM + DiscoverAirdrie Local News.\nWant to win a truck? Our Ram Everyday Adventure Contest is BACK!\n\n@RadioClaireMC
3663                                                                  hier schreiben die Hühner aus dem Stall  (*25./26.4. 2019) . \nKörner und Würmer immer willkommen
4568    #openaccess journal published by@MDPIOpenAccess, covering aspects of #landscience. IF 2021: 3.905; CiteScore 2021: 3.2.\n#LULC #landscape #soil #urban #climate
4626  Hoping hardcore twitter 2.0 lets me express honest opinions about Danielle Smith's supporters.\n\n"Let them see what is on the end of that long newspaper spoon."
5113                                                                            

<h2>6 records contain a decimal number</h2>

**Reference List (APA 7):**
OpenAI. (2025). ChatGPT (May 17 version) [Large language model]. https://chat.openai.com/  


First Prompt: "How to use regex to find decimal numbers"   
Last Prompt : "records 3663, and 4316 has to be cleaned, without affecting other records as they are date formats, how to handle that"